In [2]:
# Install required dependencies
# Run once, can be skipped if already installed
!pip install openai -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from openai import OpenAI

client = OpenAI(
    api_key="your LLM KEY",
    base_url="https://api.siliconflow.cn/v1"
)

# 测试连通性
response = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-V3",
    messages=[{"role": "user", "content": "你好，回复'连接成功'三个字"}],
    max_tokens=20
)

print(response.choices[0].message.content)

连接成功


In [4]:
import sqlite3
import pandas as pd
DB_PATH = "C:/Users/许佳佳/Desktop/UK_Smart_Retail_Data_Ops/database/olist_dw.db"
def get_schema(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 获取所有表名
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [row[0] for row in cursor.fetchall()]

    schema_text = ""
    for table in tables:
        cursor.execute(f"PRAGMA table_info({table})")
        columns = cursor.fetchall()
        col_names = [f"{col[1]} ({col[2]})" for col in columns]
        schema_text += f"Table: {table}\nColumns: {', '.join(col_names)}\n\n"

    conn.close()
    return schema_text

schema = get_schema(DB_PATH)
print(schema)

Table: dim_customers
Columns: customer_id (TEXT), customer_unique_id (TEXT), customer_zip_code_prefix (BIGINT), customer_city (TEXT), customer_state (TEXT)

Table: dim_geolocation
Columns: geolocation_zip_code_prefix (BIGINT), geolocation_lat (FLOAT), geolocation_lng (FLOAT), geolocation_city (TEXT), geolocation_state (TEXT)

Table: dim_products
Columns: product_id (TEXT), product_category_name (TEXT), product_name_lenght (FLOAT), product_description_lenght (FLOAT), product_photos_qty (FLOAT), product_weight_g (FLOAT), product_length_cm (FLOAT), product_height_cm (FLOAT), product_width_cm (FLOAT), product_category_name_english (TEXT)

Table: dim_sellers
Columns: seller_id (TEXT), seller_zip_code_prefix (BIGINT), seller_city (TEXT), seller_state (TEXT)

Table: dim_orders
Columns: order_id (TEXT), customer_id (TEXT), order_status (TEXT), order_purchase_timestamp (TEXT), order_approved_at (TEXT), order_delivered_carrier_date (TEXT), order_delivered_customer_date (TEXT), order_estimated_de

In [5]:
import sqlite3
import pandas as pd

def generate_sql(user_question, schema):
    prompt = f"""你是一个数据分析专家。根据以下数据库表结构，将用户问题转换为SQLite SQL查询语句。

数据库表结构：
{schema}

重要说明：
- fact_order_items 是事实表，包含订单明细
- dim_orders 包含订单状态，已完成的订单 order_status = 'delivered'
- dim_products 包含商品信息和品类
- dim_customers 包含客户信息
- dim_sellers 包含卖家信息

用户问题：{user_question}

请只返回SQL语句，不要任何解释，不要markdown格式，直接返回可执行的SQL。"""

    response = client.chat.completions.create(
        model="deepseek-ai/DeepSeek-V3",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500,
        temperature=0
    )
    return response.choices[0].message.content.strip()


def execute_sql(sql, db_path):
    conn = sqlite3.connect(db_path)
    try:
        df = pd.read_sql_query(sql, conn)
        return df, None
    except Exception as e:
        return None, str(e)
    finally:
        conn.close()


def generate_answer(user_question, sql, df):
    data_str = df.to_string(index=False) if df is not None else "无数据"

    prompt = f"""用户问题：{user_question}
执行的SQL：{sql}
查询结果：
{data_str}

请用简洁的中文回答用户的问题，结合数据给出清晰的业务洞察，2-3句话即可。"""

    response = client.chat.completions.create(
        model="deepseek-ai/DeepSeek-V3",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300
    )
    return response.choices[0].message.content.strip()


def ask(question):
    print(f"❓ 问题：{question}")
    print("-" * 50)
    sql = generate_sql(question, schema)
    print(f"🔍 生成的SQL：\n{sql}\n")
    df, error = execute_sql(sql, DB_PATH)
    if error:
        print(f"❌ SQL执行错误：{error}")
        return
    print(f"📊 查询结果（前5行）：\n{df.head()}\n")
    answer = generate_answer(question, sql, df)
    print(f"💡 AI回答：{answer}")

print("✅ 函数定义完成，可以开始提问了")

✅ 函数定义完成，可以开始提问了


In [6]:
ask("哪个商品品类的销售额最高？")

❓ 问题：哪个商品品类的销售额最高？
--------------------------------------------------
🔍 生成的SQL：
SELECT p.product_category_name_english, SUM(oi.price) as total_sales
FROM fact_order_items oi
JOIN dim_products p ON oi.product_id = p.product_id
JOIN dim_orders o ON oi.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name_english
ORDER BY total_sales DESC
LIMIT 1;

📊 查询结果（前5行）：
  product_category_name_english  total_sales
0                 health_beauty   1233131.72

💡 AI回答：根据查询结果，健康美容品类(health_beauty)以1,233,131.72元的销售额成为最高销售品类。这表明消费者对健康和美容产品的需求旺盛，建议企业可优先加强该品类的库存管理和营销推广。


In [7]:
ask("卖家绩效最好的前5名是谁？")


❓ 问题：卖家绩效最好的前5名是谁？
--------------------------------------------------
🔍 生成的SQL：
SELECT 
    s.seller_id,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(oi.price + oi.freight_value) AS total_revenue,
    AVG(r.review_score) AS avg_review_score
FROM 
    dim_sellers s
JOIN 
    fact_order_items oi ON s.seller_id = oi.seller_id
JOIN 
    dim_orders o ON oi.order_id = o.order_id
LEFT JOIN 
    dim_reviews r ON o.order_id = r.order_id
WHERE 
    o.order_status = 'delivered'
GROUP BY 
    s.seller_id
ORDER BY 
    total_orders DESC,
    total_revenue DESC,
    avg_review_score DESC
LIMIT 5;

📊 查询结果（前5行）：
                          seller_id  total_orders  total_revenue  \
0  6560211a19b47992c3666cc44a7e94c0          1819      148421.30   
1  4a3ca9315b744ce9f8e9374361493884          1772      234120.78   
2  cc419e0650a3c5ba77189a1882b7556a          1651      128590.95   
3  1f50f920176fa81dab994f9023523100          1399      142364.99   
4  da8622b14eb17ae2831f4ac5b9dab84a          

In [8]:
ask("复购客户平均多久会再次购买？")

❓ 问题：复购客户平均多久会再次购买？
--------------------------------------------------
🔍 生成的SQL：
WITH customer_orders AS (
    SELECT 
        customer_id,
        COUNT(DISTINCT order_id) AS order_count,
        MIN(order_purchase_timestamp) AS first_order_date,
        MAX(order_purchase_timestamp) AS last_order_date
    FROM dim_orders
    WHERE order_status = 'delivered'
    GROUP BY customer_id
    HAVING COUNT(DISTINCT order_id) > 1
)

SELECT 
    AVG(julianday(last_order_date) - julianday(first_order_date)) / (order_count - 1) AS avg_days_between_orders
FROM customer_orders;

📊 查询结果（前5行）：
  avg_days_between_orders
0                    None

💡 AI回答：根据SQL查询结果（显示为None），当前数据中没有找到有多次购买记录的复购客户。这表明在分析的数据集中，所有客户均为单次购买，暂无复购行为发生。建议检查数据质量或评估是否需要扩展分析的时间范围来捕捉潜在的复购情况。


In [9]:
# Cell 5 - 交互式问答（演示用）
print("🤖 Olist 智能数据助手")
print("=" * 50)
print("你可以用自然语言查询数据库，输入 'quit' 退出\n")

while True:
    question = input("请输入你的问题：").strip()
    if question.lower() == 'quit':
        print("再见！")
        break
    if not question:
        continue
    print()
    ask(question)
    print()

🤖 Olist 智能数据助手
你可以用自然语言查询数据库，输入 'quit' 退出


❓ 问题：哪个州的客户消费最高？
--------------------------------------------------
🔍 生成的SQL：
SELECT 
    c.customer_state,
    SUM(oi.price + oi.freight_value) AS total_spending
FROM 
    fact_order_items oi
JOIN 
    dim_orders o ON oi.order_id = o.order_id
JOIN 
    dim_customers c ON o.customer_id = c.customer_id
WHERE 
    o.order_status = 'delivered'
GROUP BY 
    c.customer_state
ORDER BY 
    total_spending DESC
LIMIT 1;

📊 查询结果（前5行）：
  customer_state  total_spending
0             SP      5769703.15

💡 AI回答：查询结果显示，圣保罗州（SP）的客户消费最高，总消费金额达到576.97万元。这表明该州是业务最重要的市场区域，需要重点关注该地区的客户维护和市场策略。


❓ 问题：评分最低的商品品类是什么？
--------------------------------------------------
🔍 生成的SQL：
SELECT p.product_category_name_english, AVG(r.review_score) as avg_score
FROM dim_products p
JOIN fact_order_items oi ON p.product_id = oi.product_id
JOIN dim_orders o ON oi.order_id = o.order_id
JOIN dim_reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
GROUP BY p.prod

KeyboardInterrupt: Interrupted by user